# Install & Imports

In [ ]:
!pip install merlinquantum datasets

from datasets import load_dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import merlin as ML
from merlin import QuantumLayer, LexGrouping
from merlin.builder import CircuitBuilder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV


# Load Data

In [ ]:
ds_level1 = load_dataset(
    "Quandela/Challenge_Swaptions",
    data_files="level-1_Future_prediction/train.csv",
    split="train",
)
df = pd.DataFrame(ds_level1)
df_numeric = df.drop(columns=['Date'])
print("Shape:", df_numeric.shape)

# Visualize Data


In [ ]:
cols_to_plot = df_numeric.columns[:5]
df_numeric[cols_to_plot].plot(figsize=(14, 5), title="Swaption Price Time Series (first 5 features)")
plt.tight_layout()
plt.savefig("timeseries.png")
plt.show()
print("Number of swaption features:", df_numeric.shape[1])
print("Number of time steps (rows):", df_numeric.shape[0])


# Preprocessing


In this cell, we prepare the time series data for supervised learning.
First, we create sliding window sequences so the model uses the previous 3 time steps to predict the next one.
Then, we standardize the data and split it into training (80%) and testing (20%) sets while preserving temporal order.
Finally, we check the shapes and extract input/output dimensions for model configuration.



In [ ]:
def make_sequences(data, lookback=3):
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i-lookback:i])
        y.append(data[i])
    return np.array(X), np.array(y)

values = df_numeric.values.astype("float32")
scaler = StandardScaler()
values_scaled = scaler.fit_transform(values)

LOOKBACK = 3
X, y = make_sequences(values_scaled, lookback=LOOKBACK)
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
n_seq_features = X_train.shape[2]
n_outputs      = y_train.shape[1]


# Quantum Circuit

In this cell, we define the quantum feature extractor architecture.
We build a small photonic circuit (with encoding, entangling layers, and rotations) and wrap it as a QuantumLayer, followed by a feature grouping layer.
Then, we create an ensemble of 3 quantum circuits with different random seeds to enrich feature diversity.
Finally, we compute the total number of extracted features per sample.



In [ ]:
N_MODES      = 6
N_PHOTONS    = 3
N_QUANTUM_IN = 4
GROUPING_DIM = 16

# Same random projection as working version
torch.manual_seed(42)
proj_matrix = torch.randn(n_seq_features, N_QUANTUM_IN) * 0.1

def build_extractor():
    builder = CircuitBuilder(n_modes=N_MODES)
    builder.add_entangling_layer(trainable=True, name="U1")
    builder.add_angle_encoding(modes=list(range(N_QUANTUM_IN)), name="input")
    builder.add_rotations(trainable=True, name="theta")
    builder.add_superpositions(depth=1)
    core = QuantumLayer(
        input_size=N_QUANTUM_IN,
        builder=builder,
        n_photons=N_PHOTONS,
        dtype=torch.float32,
    )
    grouping = LexGrouping(core.output_size, GROUPING_DIM)
    extractor = nn.Sequential(core, grouping)
    extractor.eval()
    return extractor

# TWEAK 1: Build 3 circuits with different seeds, concatenate features
# This is the QR2 ensemble idea but conservative — same small circuit
extractors = []
for seed in [42, 7, 123]:
    torch.manual_seed(seed)
    extractors.append(build_extractor())

print(f"Built {len(extractors)} quantum circuits")
print(f"Total features per sample: {LOOKBACK * GROUPING_DIM * len(extractors)}")

# Feature extraction

In this cell, we extract quantum features from the time-series data.
Each timestep is first projected to a low-dimensional quantum input space, then passed through multiple quantum circuits to generate features.
Features from all circuits and timesteps are concatenated into a single vector per sample.
Finally, we compute and store the quantum features for both training and test sets.

In [ ]:
def extract_quantum_features(X_np):
    """
    X_np: (B, T, F)
    Returns: (B, T * GROUPING_DIM * n_circuits)
    """
    B, T, F = X_np.shape
    X_t = torch.tensor(X_np, dtype=torch.float32)

    all_features = []
    for t in range(T):
        xt     = X_t[:, t, :]           # (B, F)
        xt_proj = torch.sigmoid(xt @ proj_matrix)  # (B, N_QUANTUM_IN) in [0,1]

        timestep_feats = []
        for ext in extractors:
            with torch.no_grad():
                qf = ext(xt_proj)       # (B, GROUPING_DIM)
            timestep_feats.append(qf)

        all_features.append(torch.cat(timestep_feats, dim=1))  # (B, GROUPING_DIM*3)

    return torch.cat(all_features, dim=1).numpy()  # (B, T * GROUPING_DIM * 3)

print("Extracting training features...")
Q_train = extract_quantum_features(X_train)
print("Extracting test features...")
Q_test  = extract_quantum_features(X_test)
print(f"Feature shape: {Q_train.shape}")

# Tweek
In this cell, we train a Ridge regression model on the extracted quantum features.
We use cross-validation to automatically select the best regularization strength (alpha).
After training, we generate predictions for both the training and test sets.

In [ ]:
alphas = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]
ridge = RidgeCV(alphas=alphas, cv=5)
ridge.fit(Q_train, y_train)
print(f"Best alpha: {ridge.alpha_}")

y_pred_test  = ridge.predict(Q_test)
y_pred_train = ridge.predict(Q_train)

# MLP and LSTM Baselines
In this cell, we prepare PyTorch tensors for both flattened and sequential inputs.
We define a generic training loop using Adam and a learning rate scheduler.
Then, we implement two classical baseline models: an MLP (using flattened inputs) and an LSTM (using sequential inputs).
Finally, we train both models to compare their performance against the quantum-based approach.

In [ ]:
X_tr_flat = torch.tensor(X_train.reshape(len(X_train), -1), dtype=torch.float32)
X_te_flat = torch.tensor(X_test.reshape(len(X_test),   -1), dtype=torch.float32)
X_tr_seq  = torch.tensor(X_train, dtype=torch.float32)
X_te_seq  = torch.tensor(X_test,  dtype=torch.float32)
y_tr = torch.tensor(y_train, dtype=torch.float32)
y_te = torch.tensor(y_test,  dtype=torch.float32)
n_flat_features = X_tr_flat.shape[1]

def train_model(model, X_tr, y_tr, epochs=200, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=80, gamma=0.5)
    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        loss = F.mse_loss(model(X_tr), y_tr)
        loss.backward()
        opt.step()
        scheduler.step()
        if ep % 50 == 0:
            print(f"  Epoch {ep:3d}: loss={loss.item():.6f}")

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_flat_features, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, n_outputs)
        )
    def forward(self, x): return self.net(x)

class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(n_seq_features, hidden_size=64, num_layers=2, batch_first=True)
        self.head = nn.Linear(64, n_outputs)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])

print("Training MLP..."); mlp = MLP(); train_model(mlp, X_tr_flat, y_tr)
print("Training LSTM..."); lstm_model = LSTMModel(); train_model(lstm_model, X_tr_seq, y_tr)

# Metrics:
In this cell, we define evaluation metrics to assess model performance.
Predictions are first transformed back to the original scale before computing RMSE, MAE, and QLIKE.
We create separate evaluation functions for NumPy-based (Ridge) and PyTorch models.
Finally, we compare the Quantum Ridge model with the MLP and LSTM baselines on the test set.

In [ ]:
def compute_metrics_np(preds_scaled, true_scaled):
    preds_orig = scaler.inverse_transform(preds_scaled)
    true_orig  = scaler.inverse_transform(true_scaled)
    rmse  = np.sqrt(np.mean((preds_orig - true_orig)**2))
    mae   = np.mean(np.abs(preds_orig - true_orig))
    eps   = 1e-8
    qlike = np.mean(np.log(true_orig**2 + eps) + preds_orig / (true_orig + eps))
    return {"RMSE": round(rmse,6), "MAE": round(mae,6), "QLIKE": round(qlike,6)}

def compute_metrics_torch(model, X_te, y_te):
    model.eval()
    with torch.no_grad():
        preds = model(X_te).numpy()
    return compute_metrics_np(preds, y_te.numpy())

print("\n===== RESULTS =====")
print("QRC (3-circuit ensemble + RidgeCV):", compute_metrics_np(y_pred_test, y_test))
print("MLP Baseline:                      ", compute_metrics_torch(mlp,        X_te_flat, y_te))
print("LSTM Baseline:                     ", compute_metrics_torch(lstm_model, X_te_seq,  y_te))



# Plot
In this cell, we visualize and compare the predictions of all models.
We first convert predictions back to the original scale.
Then, we plot actual vs predicted values for the Quantum model, MLP, and LSTM in separate subplots.
Finally, we save the figure to compare model performance visually.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))
true_orig = scaler.inverse_transform(y_test)

mlp.eval(); lstm_model.eval()
with torch.no_grad():
    mlp_preds  = mlp(X_te_flat).numpy()
    lstm_preds = lstm_model(X_te_seq).numpy()

for ax, (preds_scaled, name) in zip(axes, [
    (y_pred_test, "QRC 3-Circuit Ensemble + RidgeCV"),
    (mlp_preds,   "MLP Baseline"),
    (lstm_preds,  "LSTM Baseline"),
]):
    pred_orig = scaler.inverse_transform(preds_scaled)
    ax.plot(true_orig[:, 0], label="Actual", linewidth=2)
    ax.plot(pred_orig[:, 0], label=name,     linestyle="--")
    ax.set_title(name)
    ax.legend()

plt.tight_layout()
plt.savefig("all_predictions.png")
plt.show()


# Prediction
In this cell, we prepare the unseen test data for final prediction.
We scale it using the same scaler and add the last training timesteps to preserve temporal context.
Then, we generate sequences, extract quantum features, and produce predictions using the trained Ridge model.
Finally, we convert predictions back to the original scale and save them as submission.csv.

In [ ]:
df_test = pd.read_excel("/content/test.xlsx")
if 'Date' in df_test.columns:
    df_test = df_test.drop(columns=['Date'])

test_vals   = df_test.values.astype("float32")
test_scaled = scaler.transform(test_vals)

last_train_rows          = values_scaled[-LOOKBACK:]
test_scaled_with_context = np.concatenate([last_train_rows, test_scaled], axis=0)

X_final, _ = make_sequences(test_scaled_with_context, lookback=LOOKBACK)
print(f"X_final shape: {X_final.shape}")  # (6, 3, F)

Q_final      = extract_quantum_features(X_final)
final_preds  = ridge.predict(Q_final)

final_preds_orig = scaler.inverse_transform(final_preds)
submission = pd.DataFrame(final_preds_orig, columns=df_numeric.columns)
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv!")
print(submission.round(6))

# Deployment

In [ ]:
!pip install gradio -q
import gradio as gr
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import io, base64
from sklearn.preprocessing import StandardScaler
import torch, torch.nn.functional as F

def _inverse(arr):
    return scaler.inverse_transform(arr)

def _metrics(pred_s, true_s):
    p, t = _inverse(pred_s), _inverse(true_s)
    rmse  = float(np.sqrt(np.mean((p - t)**2)))
    mae   = float(np.mean(np.abs(p - t)))
    eps   = 1e-8
    qlike = float(np.mean(np.log(t**2 + eps) + p / (t + eps)))
    return rmse, mae, qlike

def show_metrics():
    rows = []
    qp = ridge.predict(extract_quantum_features(X_test))
    r, m, q = _metrics(qp, y_test)
    rows.append({"Model": "⚛️  QRC Ensemble + RidgeCV", "RMSE": r, "MAE": m, "QLIKE": q})
    mlp.eval()
    with torch.no_grad():
        mp = mlp(X_te_flat).numpy()
    r, m, q = _metrics(mp, y_test)
    rows.append({"Model": "🧠  MLP Baseline", "RMSE": r, "MAE": m, "QLIKE": q})
    lstm_model.eval()
    with torch.no_grad():
        lp = lstm_model(X_te_seq).numpy()
    r, m, q = _metrics(lp, y_test)
    rows.append({"Model": "🔁  LSTM Baseline", "RMSE": r, "MAE": m, "QLIKE": q})
    return pd.DataFrame(rows).set_index("Model").round(6)

def comparison_plot(feature_idx: int):
    feature_idx = int(feature_idx) - 1
    feature_idx = max(0, min(feature_idx, y_test.shape[1] - 1))
    true_orig = _inverse(y_test)
    qp = ridge.predict(extract_quantum_features(X_test))
    q_orig = _inverse(qp)
    mlp.eval(); lstm_model.eval()
    with torch.no_grad():
        mp = mlp(X_te_flat).numpy()
        lp = lstm_model(X_te_seq).numpy()
    m_orig = _inverse(mp)
    l_orig = _inverse(lp)
    fig, axes = plt.subplots(3, 1, figsize=(13, 11), facecolor="#0f0f1a")
    palette = {"actual": "#e8e0d0", "qrc": "#7efff5", "mlp": "#ff7eb3", "lstm": "#ffd97d"}
    titles = ["⚛️  QRC Ensemble + RidgeCV", "🧠  MLP Baseline", "🔁  LSTM Baseline"]
    preds  = [q_orig, m_orig, l_orig]
    clrs   = [palette["qrc"], palette["mlp"], palette["lstm"]]
    for ax, pred, title, clr in zip(axes, preds, titles, clrs):
        ax.set_facecolor("#14142b")
        ax.plot(true_orig[:, feature_idx], color=palette["actual"], linewidth=1.8, label="Actual", alpha=0.9)
        ax.plot(pred[:, feature_idx], color=clr, linewidth=1.5, linestyle="--", label=title, alpha=0.92)
        ax.set_title(title, color="#e8e0d0", fontsize=12, pad=8)
        ax.tick_params(colors="#888")
        ax.spines[["top","right","left","bottom"]].set_color("#333")
        ax.legend(facecolor="#1e1e3a", edgecolor="#444", labelcolor="#e8e0d0", fontsize=9)
        ax.set_xlabel(f"Feature #{feature_idx+1} — Test timestep", color="#888", fontsize=9)
    fig.suptitle(f"Model Comparison — Swaption Feature #{feature_idx+1}", color="#e8e0d0", fontsize=14, y=1.01)
    plt.tight_layout()
    return fig

def predict_from_upload(file):
    if file is None:
        return "⚠️ Please upload a file.", None
    try:
        if file.name.endswith(".csv"):
            df_up = pd.read_csv(file.name)
        else:
            df_up = pd.read_excel(file.name)
        if "Date" in df_up.columns:
            df_up = df_up.drop(columns=["Date"])
        test_vals   = df_up.values.astype("float32")
        test_scaled = scaler.transform(test_vals)
        last_rows   = values_scaled[-LOOKBACK:]
        combined    = np.concatenate([last_rows, test_scaled], axis=0)
        X_final, _  = make_sequences(combined, lookback=LOOKBACK)
        Q_final     = extract_quantum_features(X_final)
        preds       = ridge.predict(Q_final)
        preds_orig  = scaler.inverse_transform(preds)
        df_result   = pd.DataFrame(preds_orig, columns=df_numeric.columns)
        return "✅ Predictions generated successfully!", df_result.round(6)
    except Exception as e:
        return f"❌ Error: {str(e)}", None

def retrain_ridge(alpha_val: float):
    from sklearn.linear_model import Ridge
    alpha_val = float(alpha_val)
    Q_tr = extract_quantum_features(X_train)
    Q_te = extract_quantum_features(X_test)
    new_ridge = Ridge(alpha=alpha_val)
    new_ridge.fit(Q_tr, y_train)
    preds = new_ridge.predict(Q_te)
    r, m, q   = _metrics(preds, y_test)
    curr      = ridge.predict(Q_te)
    cr, cm, cq = _metrics(curr, y_test)
    result = (
        f"### Ridge α={alpha_val:.4f}\n"
        f"- **RMSE:** {r:.6f}  (current best: {cr:.6f})\n"
        f"- **MAE:**  {m:.6f}  (current best: {cm:.6f})\n"
        f"- **QLIKE:** {q:.6f}  (current best: {cq:.6f})\n"
    )
    return result + ("\n🟢 Improvement!" if r < cr else "\n🔴 Worse than current best.")

with gr.Blocks(
    title="⚛️ Quantum Swaption Predictor",
    theme=gr.themes.Base(primary_hue="cyan", neutral_hue="slate", font=gr.themes.GoogleFont("IBM Plex Mono")),
    css="body { background: #0f0f1a; } .gradio-container { background: #0f0f1a !important; }",
) as demo:
    gr.Markdown("# ⚛️ Quantum Swaption Predictor\n**QRC Ensemble + Ridge | MLP | LSTM** — Hackathon Dashboard")
    with gr.Tabs():
        with gr.Tab("📊 Metrics"):
            gr.Markdown("Compute RMSE / MAE / QLIKE on the held-out test set for all three models.")
            run_btn = gr.Button("▶ Run Evaluation", variant="primary")
            metrics_table = gr.Dataframe(label="Test-set Metrics")
            run_btn.click(fn=show_metrics, outputs=metrics_table)
        with gr.Tab("📈 Comparison Plot"):
            gr.Markdown("Plot actual vs predicted for a chosen swaption feature index.")
            feat_slider = gr.Slider(minimum=1, maximum=int(y_test.shape[1]), step=1, value=1, label="Feature Index")
            plot_btn = gr.Button("▶ Plot", variant="primary")
            plot_out = gr.Plot()
            plot_btn.click(fn=comparison_plot, inputs=feat_slider, outputs=plot_out)
        with gr.Tab("📤 Upload & Predict"):
            gr.Markdown("Upload your `test.xlsx` or `test.csv` to generate a submission.")
            file_in    = gr.File(label="Upload test file (.xlsx / .csv)", file_types=[".xlsx", ".csv"])
            pred_btn   = gr.Button("▶ Predict", variant="primary")
            status     = gr.Markdown()
            pred_table = gr.Dataframe(label="Predictions (original scale)")
            pred_btn.click(fn=predict_from_upload, inputs=file_in, outputs=[status, pred_table])
        with gr.Tab("🔧 Tune Ridge α"):
            gr.Markdown("Try a different Ridge regularisation strength and see the impact instantly.")
            alpha_in = gr.Number(value=0.1, label="Ridge alpha (e.g. 0.001 → 10.0)")
            tune_btn = gr.Button("▶ Retrain & Compare", variant="primary")
            tune_out = gr.Markdown()
            tune_btn.click(fn=retrain_ridge, inputs=alpha_in, outputs=tune_out)

demo.launch(share=True)